# Satellite Phenology Refinement (6 May 2026)

This notebook rebuilds the SIT satellite pipeline with a focus on cleaner, more usable signals for downstream phenology modelling.

What this notebook does:
- Uses the current SIT consensus crowns as the spatial unit.
- Drops known non-tree crowns when the SIT phenology drop-list is available.
- Queries Sentinel-2 L2A over the AOI for 2022-2024.
- Computes per-crown NDVI and EVI with Sentinel Scene Classification (SCL) masking for cloud, shadow, snow, water, and no-data pixels.
- Saves all plots and long tables to disk instead of rendering large outputs inline.
- Builds weekly and monthly summaries, grouped diagnostics, and label-linked satellite features for the next ML step.

The notebook keeps inline output intentionally short. Inspect the saved PNG/CSV/JSON artifacts in the run folder for detailed results.

In [8]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from shapely.geometry import box, mapping
from pystac_client import Client
import planetary_computer as pc
import rasterio
from rasterio import features
from rasterio.enums import Resampling
from pyproj import Transformer

warnings.filterwarnings('ignore', category=UserWarning)

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'output').exists() and (candidate / 'src').exists():
            return candidate
        if (candidate / 'Readme.md').exists() or (candidate / 'README.md').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
OUTPUT_ROOT = REPO_ROOT / 'output'
NOTEBOOK_NAME = 'sat_data6May26'
RUN_LABEL = 'sit_scl_masked_6May26'
RUN_DIR = OUTPUT_ROOT / 'sat_data' / RUN_LABEL
PLOTS_DIR = RUN_DIR / 'plots'
TABLES_DIR = RUN_DIR / 'tables'
META_DIR = RUN_DIR / 'meta'
for folder in [RUN_DIR, PLOTS_DIR, TABLES_DIR, META_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'sit'
YEARS = [2022, 2023, 2024]
START_MM_DD = '01-01'
END_MM_DD = '12-31'
MAX_CLOUD_PCT_USE = 40
ONE_ITEM_PER_DATE = True
CROWN_SAMPLE_SIZE = 20
CROWN_SAMPLE_RANDOM_SEED = 26
MIN_VALID_PIXELS_RELAXED = 1
MIN_VALID_PIXELS_STRICT = 3
ALL_TOUCHED = True
PAD_PIXELS = 40
SCL_INVALID_CLASSES = {0, 1, 3, 6, 8, 9, 10, 11}
STAC_API = 'https://planetarycomputer.microsoft.com/api/stac/v1'
COLLECTION = 'sentinel-2-l2a'

SIT_CROWNS_CANDIDATES = [
    OUTPUT_ROOT / 'sit_tracking_rerun_1Apr26' / 'consensus_crowns_om1_phenology_sit.geojson',
    OUTPUT_ROOT / 'consensus_crowns_om1_phenology.geojson',
    OUTPUT_ROOT / 'consensus_crowns_tree_only.gpkg',
]
SIT_NON_TREE_DROP_CANDIDATES = [
    OUTPUT_ROOT / 'sit_tracking_rerun_1Apr26' / 'phenology' / 'non_tree_dropped_crowns.csv',
    OUTPUT_ROOT / 'non_tree_chain_ids.csv',
]
SIT_LABEL_CANDIDATES = [
    OUTPUT_ROOT / 'sit_tracking_rerun_1Apr26' / 'phenology' / 'leafshed_tree_scores.csv',
    OUTPUT_ROOT / 'leaf_shed_classification.csv',
]

def first_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None

crowns_path = first_existing(SIT_CROWNS_CANDIDATES)
drop_path = first_existing(SIT_NON_TREE_DROP_CANDIDATES)
labels_path = first_existing(SIT_LABEL_CANDIDATES)
if crowns_path is None:
    raise FileNotFoundError('Could not find a SIT consensus crowns file.')

print('REPO_ROOT:', REPO_ROOT)
print('RUN_DIR  :', RUN_DIR)
print('Crowns   :', crowns_path)
print('Drop list:', drop_path if drop_path else 'none')
print('Labels   :', labels_path if labels_path else 'none')
print('Sample n :', CROWN_SAMPLE_SIZE)

REPO_ROOT: /Users/hbot07/VS Code/Drone-Phenology-Monitoring
RUN_DIR  : /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_scl_masked_6May26
Crowns   : /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_1Apr26/consensus_crowns_om1_phenology_sit.geojson
Drop list: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_1Apr26/phenology/non_tree_dropped_crowns.csv
Labels   : /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_1Apr26/phenology/leafshed_tree_scores.csv
Sample n : 20


In [9]:
crowns = gpd.read_file(crowns_path).copy()
if 'chain_id' not in crowns.columns:
    raise ValueError(f"Expected 'chain_id' in crowns file, found: {list(crowns.columns)}")

crowns['chain_id'] = pd.to_numeric(crowns['chain_id'], errors='coerce')
crowns = crowns.dropna(subset=['chain_id']).copy()
crowns['chain_id'] = crowns['chain_id'].astype(int)
crowns = crowns[crowns.geometry.notnull() & ~crowns.geometry.is_empty].copy()
crowns['crown_area'] = crowns.geometry.area.astype(float)

drop_ids: set[int] = set()
if drop_path is not None:
    drop_df = pd.read_csv(drop_path)
    if 'chain_id' in drop_df.columns:
        drop_ids = set(pd.to_numeric(drop_df['chain_id'], errors='coerce').dropna().astype(int).tolist())

crowns['is_dropped_non_tree'] = crowns['chain_id'].isin(drop_ids)
crowns_filtered = crowns.loc[~crowns['is_dropped_non_tree']].copy().reset_index(drop=True)

labels_df = pd.DataFrame()
if labels_path is not None:
    labels_df = pd.read_csv(labels_path).copy()
    if 'chain_id' in labels_df.columns:
        labels_df['chain_id'] = pd.to_numeric(labels_df['chain_id'], errors='coerce')
        labels_df = labels_df.dropna(subset=['chain_id']).copy()
        labels_df['chain_id'] = labels_df['chain_id'].astype(int)
        labels_df = labels_df.drop_duplicates(subset=['chain_id']).copy()

label_cols = [
    'chain_id', 'is_deciduous', 'deciduous_score', 'leaf_off_start_om',
    'full_leaf_off_om', 'leaf_on_return_om', 'is_non_tree', 'is_tree'
 ]
available_label_cols = [col for col in label_cols if col in labels_df.columns]
if available_label_cols:
    crowns_filtered = crowns_filtered.merge(
        labels_df[available_label_cols],
        on='chain_id',
        how='left',
    )

def select_diverse_crowns(crowns_gdf: gpd.GeoDataFrame, sample_size: int, seed: int) -> gpd.GeoDataFrame:
    if len(crowns_gdf) <= sample_size:
        out = crowns_gdf.copy().reset_index(drop=True)
        out['sample_group'] = 'all'
        return out

    work = crowns_gdf.copy()
    work['is_deciduous_group'] = work.get('is_deciduous', pd.Series(False, index=work.index)).fillna(False).astype(bool)
    unique_areas = max(1, work['crown_area'].nunique())
    n_area_bins = min(4, unique_areas)
    if n_area_bins > 1:
        work['area_bin'] = pd.qcut(work['crown_area'], q=n_area_bins, duplicates='drop').astype(str)
    else:
        work['area_bin'] = 'all_areas'
    work['sample_group'] = np.where(work['is_deciduous_group'], 'deciduous', 'other') + ' | ' + work['area_bin'].astype(str)

    groups = []
    rng = np.random.default_rng(seed)
    grouped = list(work.groupby('sample_group', dropna=False))
    base_take = max(1, sample_size // max(len(grouped), 1))

    taken_ids: list[int] = []
    for _, group in grouped:
        take_n = min(len(group), base_take)
        if take_n == 0:
            continue
        sample_idx = rng.choice(group.index.to_numpy(), size=take_n, replace=False)
        taken_ids.extend(sample_idx.tolist())

    selected = work.loc[sorted(set(taken_ids))].copy()
    if len(selected) < sample_size:
        remaining = work.drop(index=selected.index)
        topup_n = min(sample_size - len(selected), len(remaining))
        if topup_n > 0:
            topup_idx = rng.choice(remaining.index.to_numpy(), size=topup_n, replace=False)
            selected = pd.concat([selected, work.loc[topup_idx].copy()], axis=0)

    selected = selected.sort_values(['sample_group', 'crown_area', 'chain_id']).head(sample_size).reset_index(drop=True)
    return selected

crowns_sampled = select_diverse_crowns(
    crowns_filtered,
    sample_size=CROWN_SAMPLE_SIZE,
    seed=CROWN_SAMPLE_RANDOM_SEED,
).copy()

bounds = crowns_sampled.total_bounds
aoi_poly = box(*bounds)
aoi = gpd.GeoDataFrame({'name': ['aoi']}, geometry=[aoi_poly], crs=crowns_sampled.crs)
aoi_wgs84 = aoi.to_crs('EPSG:4326')
bbox_wgs84 = [float(x) for x in aoi_wgs84.total_bounds]
centroid_wgs84 = aoi_wgs84.geometry.iloc[0].centroid

crowns_export = RUN_DIR / 'crowns_filtered_sample.geojson'
crowns_sampled.to_file(crowns_export, driver='GeoJSON')

sample_summary = (
    crowns_sampled.groupby('sample_group', dropna=False)
    .agg(n_crowns=('chain_id', 'size'), median_area=('crown_area', 'median'))
    .reset_index()
    .sort_values('sample_group')
)
sample_summary.to_csv(TABLES_DIR / 'crown_sample_summary.csv', index=False)

run_meta = {
    'dataset_name': DATASET_NAME,
    'years': YEARS,
    'crowns_path': str(crowns_path),
    'drop_path': str(drop_path) if drop_path else None,
    'labels_path': str(labels_path) if labels_path else None,
    'n_crowns_raw': int(len(crowns)),
    'n_drop_ids': int(len(drop_ids)),
    'n_crowns_filtered': int(len(crowns_filtered)),
    'n_crowns_sampled': int(len(crowns_sampled)),
    'crowns_crs': str(crowns_sampled.crs),
    'bbox_wgs84': bbox_wgs84,
    'centroid_wgs84': [float(centroid_wgs84.x), float(centroid_wgs84.y)],
}
(META_DIR / 'run_meta.json').write_text(json.dumps(run_meta, indent=2), encoding='utf-8')

print(f'Loaded {len(crowns)} crowns, kept {len(crowns_filtered)} after drop-list filtering.')
print(f'Sampled {len(crowns_sampled)} diverse crowns for satellite prototyping.')
print(f'Label rows available: {len(labels_df)}')
print(f'AOI bbox WGS84: {bbox_wgs84}')
print(f'Saved sampled crowns to {crowns_export}')
print(sample_summary.to_string(index=False))

Loaded 131 crowns, kept 126 after drop-list filtering.
Sampled 20 diverse crowns for satellite prototyping.
Label rows available: 158
AOI bbox WGS84: [77.19065588450114, 28.544711067907027, 77.19232187419638, 28.546334350355384]
Saved sampled crowns to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_scl_masked_6May26/crowns_filtered_sample.geojson
                 sample_group  n_crowns  median_area
 deciduous | (25.905, 44.174]         2    38.169002
 deciduous | (44.174, 84.796]         2    50.748399
  deciduous | (5.031, 25.905]         4    13.040854
deciduous | (84.796, 562.125]         3    97.443680
     other | (25.905, 44.174]         2    37.564919
     other | (44.174, 84.796]         3    56.748276
      other | (5.031, 25.905]         2    13.396477
    other | (84.796, 562.125]         2    97.832693


/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_98977/2744979096.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  work['is_deciduous_group'] = work.get('is_deciduous', pd.Series(False, index=work.index)).fillna(False).astype(bool)


In [5]:
import re

catalog = Client.open(STAC_API)

def item_date_str(item) -> str | None:
    return item.datetime.strftime('%Y-%m-%d') if item.datetime else None

def sentinel_epsg_from_item(item, fallback_lat: float) -> int:
    epsg = item.properties.get('proj:epsg')
    if epsg is not None:
        return int(epsg)
    tile = item.properties.get('mgrs:tile')
    zone = None
    if isinstance(tile, str) and len(tile) >= 2 and tile[:2].isdigit():
        zone = int(tile[:2])
    if zone is None:
        match = re.search(r'_T(\d{2})[A-Z]{3}_', item.id)
        if match:
            zone = int(match.group(1))
    if zone is None:
        raise RuntimeError(f'Could not derive EPSG for {item.id}')
    return 32600 + zone if fallback_lat >= 0 else 32700 + zone

def apply_item_filters(df: pd.DataFrame, max_cloud: int | None, one_item_per_date: bool) -> pd.DataFrame:
    out = df.copy()
    out['cloud_cover'] = pd.to_numeric(out['cloud_cover'], errors='coerce')
    if max_cloud is not None:
        out = out[out['cloud_cover'].notna() & (out['cloud_cover'] <= float(max_cloud))].copy()
    if one_item_per_date:
        out = (
            out.sort_values(['date', 'cloud_cover', 'id'])
            .groupby('date', as_index=False)
            .first()
        )
    return out.sort_values(['date', 'cloud_cover']).reset_index(drop=True)

def query_year(year: int) -> list:
    start_date = f'{year}-{START_MM_DD}'
    end_date = f'{year}-{END_MM_DD}'
    search = catalog.search(
        collections=[COLLECTION],
        bbox=bbox_wgs84,
        datetime=f'{start_date}/{end_date}',
    )
    return list(search.get_items())

fallback_lat = float(centroid_wgs84.y)
items_by_year = {year: query_year(year) for year in YEARS}
items = [item for year in YEARS for item in items_by_year[year]]

item_rows = []
for item in items:
    dt = item_date_str(item)
    if dt is None:
        continue
    item_rows.append({
        'id': item.id,
        'datetime': dt,
        'cloud_cover': item.properties.get('eo:cloud_cover'),
        'mgrs_tile': item.properties.get('mgrs:tile'),
        'orbit_state': item.properties.get('sat:orbit_state'),
        'derived_epsg': sentinel_epsg_from_item(item, fallback_lat),
    })

items_df_raw = pd.DataFrame(item_rows)
items_df_raw['date'] = pd.to_datetime(items_df_raw['datetime'], errors='coerce')
items_df_raw = items_df_raw.dropna(subset=['date']).copy()
items_df_raw['year'] = items_df_raw['date'].dt.year.astype(int)
iso_raw = items_df_raw['date'].dt.isocalendar()
items_df_raw['iso_year'] = iso_raw.year.astype(int)
items_df_raw['iso_week'] = iso_raw.week.astype(int)

items_df = apply_item_filters(items_df_raw, MAX_CLOUD_PCT_USE, ONE_ITEM_PER_DATE)

items_df_raw.to_csv(TABLES_DIR / 'sentinel2_items_raw.csv', index=False)
items_df.to_csv(TABLES_DIR / 'sentinel2_items_filtered.csv', index=False)

coverage_rows = []
for threshold in [20, 30, 40, 50, None]:
    filtered = apply_item_filters(items_df_raw, threshold, ONE_ITEM_PER_DATE)
    coverage_rows.append({
        'cloud_threshold': 'none' if threshold is None else threshold,
        'n_items': int(len(filtered)),
        'n_unique_dates': int(filtered['date'].nunique()) if len(filtered) else 0,
        'n_iso_weeks': int(filtered[['iso_year', 'iso_week']].drop_duplicates().shape[0]) if len(filtered) else 0,
        'n_years': int(filtered['year'].nunique()) if len(filtered) else 0,
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(TABLES_DIR / 'cloud_threshold_coverage_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4), dpi=160)
ax.bar(coverage_df['cloud_threshold'].astype(str), coverage_df['n_unique_dates'], color='#2a9d8f')
ax.set_title('Sentinel-2 unique dates by cloud threshold')
ax.set_xlabel('cloud threshold')
ax.set_ylabel('unique dates')
ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'cloud_threshold_coverage.png', bbox_inches='tight')
plt.close(fig)

if items_df.empty:
    raise RuntimeError('No Sentinel-2 items left after filtering.')

best_item_id = items_df.sort_values(['cloud_cover', 'date']).iloc[0]['id']
id_to_item = {item.id: item for item in items}
best_item = id_to_item[best_item_id]
s2_epsg_common = int(items_df.iloc[0]['derived_epsg'])
s2_crs_str = f'EPSG:{s2_epsg_common}'

print(f'Queried {len(items_df_raw)} raw items and kept {len(items_df)} filtered dates.')
print(f'Years represented: {sorted(items_df["year"].unique().tolist())}')
print(f'Using Sentinel CRS {s2_crs_str}; best item is {best_item_id}.')
print(f'Saved item tables to {TABLES_DIR}')

/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Queried 294 raw items and kept 127 filtered dates.
Years represented: [2022, 2023, 2024]
Using Sentinel CRS EPSG:32643; best item is S2B_MSIL2A_20220330T052639_R105_T43RGM_20220330T170127.
Saved item tables to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_scl_masked_6May26/tables


In [10]:
def signed_href(item, asset_key: str) -> str:
    asset = item.assets.get(asset_key)
    if asset is None:
        raise KeyError(f'Missing asset {asset_key} on item {item.id}')
    return pc.sign(asset.href)

def aoi_bounds_in_crs(crs_str: str) -> tuple[float, float, float, float]:
    minlon, minlat, maxlon, maxlat = bbox_wgs84
    transformer = Transformer.from_crs('EPSG:4326', crs_str, always_xy=True)
    xs, ys = transformer.transform(
        [minlon, minlon, maxlon, maxlon],
        [minlat, maxlat, minlat, maxlat],
    )
    return float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))

def read_item_bands(item, crs_str: str):
    href_red = signed_href(item, 'B04')
    href_nir = signed_href(item, 'B08')
    href_blue = signed_href(item, 'B02')
    href_scl = signed_href(item, 'SCL')
    bounds = aoi_bounds_in_crs(crs_str)
    with rasterio.open(href_red) as src_red, rasterio.open(href_nir) as src_nir, rasterio.open(href_blue) as src_blue, rasterio.open(href_scl) as src_scl:
        window = rasterio.windows.from_bounds(*bounds, transform=src_red.transform)
        window = window.round_offsets().round_lengths()
        window = rasterio.windows.Window(
            col_off=max(0, window.col_off - PAD_PIXELS),
            row_off=max(0, window.row_off - PAD_PIXELS),
            width=min(src_red.width - max(0, window.col_off - PAD_PIXELS), window.width + 2 * PAD_PIXELS),
            height=min(src_red.height - max(0, window.row_off - PAD_PIXELS), window.height + 2 * PAD_PIXELS),
        )
        red = src_red.read(1, window=window, masked=True).astype('float32')
        nir = src_nir.read(1, window=window, masked=True).astype('float32')
        blue = src_blue.read(1, window=window, masked=True).astype('float32')
        scl = src_scl.read(
            1,
            window=window,
            out_shape=red.shape,
            resampling=Resampling.nearest,
            masked=True,
        )
        transform = rasterio.windows.transform(window, src_red.transform)
    return red, nir, blue, scl, transform

def compute_indices_and_mask(red, nir, blue, scl):
    red_arr = np.asarray(red, dtype='float32')
    nir_arr = np.asarray(nir, dtype='float32')
    blue_arr = np.asarray(blue, dtype='float32')
    scl_arr = np.asarray(scl)

    invalid = (
        np.ma.getmaskarray(red)
        | np.ma.getmaskarray(nir)
        | np.ma.getmaskarray(blue)
        | np.ma.getmaskarray(scl)
        | np.isin(scl_arr, list(SCL_INVALID_CLASSES))
    )

    ndvi_denom = nir_arr + red_arr
    evi_denom = nir_arr + 6.0 * red_arr - 7.5 * blue_arr + 1.0

    invalid |= ~np.isfinite(ndvi_denom) | (ndvi_denom == 0)
    invalid |= ~np.isfinite(evi_denom) | (evi_denom == 0)

    ndvi = np.where(invalid, np.nan, (nir_arr - red_arr) / ndvi_denom)
    evi = np.where(invalid, np.nan, 2.5 * (nir_arr - red_arr) / evi_denom)

    ndvi = ndvi.astype('float32')
    evi = evi.astype('float32')
    ndvi[(ndvi < -1.0) | (ndvi > 1.0)] = np.nan
    evi[(evi < -1.0) | (evi > 1.5)] = np.nan
    valid_mask = np.isfinite(ndvi) & np.isfinite(evi)
    return ndvi, evi, valid_mask

def build_crown_masks(crowns_gdf: gpd.GeoDataFrame, transform, out_shape: tuple[int, int]):
    masks = {}
    for row in crowns_gdf.itertuples(index=False):
        crown_id = int(row.chain_id)
        mask_geom = features.geometry_mask(
            [mapping(row.geometry)],
            out_shape=out_shape,
            transform=transform,
            invert=True,
            all_touched=ALL_TOUCHED,
        )
        masks[crown_id] = {
            'mask': mask_geom,
            'total_pixels': int(mask_geom.sum()),
        }
    return masks

def zonal_stats_for_indices(ndvi: np.ndarray, evi: np.ndarray, valid_mask: np.ndarray, crown_masks: dict[int, dict[str, object]]):
    records = []
    finite = np.isfinite(ndvi) & np.isfinite(evi) & valid_mask
    for crown_id, payload in crown_masks.items():
        mask_geom = payload['mask']
        total_pixels = int(payload['total_pixels'])
        valid_sel = mask_geom & finite
        valid_pixels = int(valid_sel.sum())
        if valid_pixels:
            ndvi_mean = float(np.nanmean(ndvi[valid_sel]))
            evi_mean = float(np.nanmean(evi[valid_sel]))
        else:
            ndvi_mean = np.nan
            evi_mean = np.nan
        records.append((crown_id, ndvi_mean, evi_mean, valid_pixels, total_pixels))
    return pd.DataFrame(records, columns=['chain_id', 'ndvi_mean', 'evi_mean', 'valid_pixels', 'total_pixels'])

def percentile_stretch(arr: np.ndarray) -> np.ndarray:
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr, dtype='float32')
    lo, hi = np.percentile(arr[finite], [2, 98])
    return np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1).astype('float32')

def save_rgb_overlay(item, crs_str: str, crowns_gdf: gpd.GeoDataFrame, out_path: Path) -> None:
    red, nir, blue, scl, transform = read_item_bands(item, crs_str)
    red_arr = np.asarray(red, dtype='float32')
    green_arr = np.asarray(nir, dtype='float32')
    blue_arr = np.asarray(blue, dtype='float32')
    rgb_scaled = np.dstack([
        percentile_stretch(red_arr),
        percentile_stretch(green_arr),
        percentile_stretch(blue_arr),
    ])

    fig, ax = plt.subplots(figsize=(9, 9), dpi=180)
    ax.imshow(rgb_scaled)
    inv = ~transform
    crowns_plot = crowns_gdf.to_crs(crs_str)
    for geom in crowns_plot.geometry:
        if geom is None or geom.is_empty:
            continue
        parts = [geom] if geom.geom_type == 'Polygon' else list(getattr(geom, 'geoms', []))
        if not parts:
            continue
        poly = max(parts, key=lambda g: g.area)
        xs, ys = poly.exterior.xy
        coords = [inv * (x, y) for x, y in zip(xs, ys)]
        ax.plot([c[0] for c in coords], [c[1] for c in coords], color='yellow', linewidth=0.7, alpha=0.85)
    ax.set_title(f'Sentinel-2 pseudo-RGB with sampled crowns\n{item.id}')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)

print('Helper functions ready.')

Helper functions ready.


In [11]:
crowns_s2 = crowns_sampled.to_crs(s2_crs_str).copy()
save_rgb_overlay(best_item, s2_crs_str, crowns_sampled, PLOTS_DIR / 'best_item_rgb_overlay.png')

records = []
items_lookup = {item.id: item for item in items}
crown_masks = None
transform_signature = None

for idx, row in enumerate(items_df.itertuples(index=False), start=1):
    item = items_lookup[row.id]
    print(f'[{idx:03d}/{len(items_df):03d}] Reading {row.id} | date={pd.Timestamp(row.date).date()} | cloud={row.cloud_cover}')
    red, nir, blue, scl, transform = read_item_bands(item, s2_crs_str)
    ndvi, evi, valid_mask = compute_indices_and_mask(red, nir, blue, scl)

    current_signature = (transform.a, transform.b, transform.c, transform.d, transform.e, transform.f, ndvi.shape)
    if crown_masks is None or current_signature != transform_signature:
        crown_masks = build_crown_masks(crowns_s2, transform, ndvi.shape)
        transform_signature = current_signature
        print(f'  Built raster masks for {len(crown_masks)} sampled crowns.')

    stats_df = zonal_stats_for_indices(ndvi, evi, valid_mask, crown_masks)
    valid_relaxed_count = int((stats_df['valid_pixels'] >= MIN_VALID_PIXELS_RELAXED).sum())
    valid_strict_count = int((stats_df['valid_pixels'] >= MIN_VALID_PIXELS_STRICT).sum())
    print(f'  Valid crowns this date: relaxed={valid_relaxed_count}, strict={valid_strict_count}')

    for rec in stats_df.itertuples(index=False):
        records.append({
            'date': row.date,
            'datetime': row.datetime,
            'year': int(row.year),
            'iso_year': int(row.iso_year),
            'iso_week': int(row.iso_week),
            'month': int(pd.Timestamp(row.date).month),
            'item_id': row.id,
            'cloud_cover': float(row.cloud_cover) if pd.notna(row.cloud_cover) else np.nan,
            'chain_id': int(rec.chain_id),
            'ndvi_mean': float(rec.ndvi_mean) if pd.notna(rec.ndvi_mean) else np.nan,
            'evi_mean': float(rec.evi_mean) if pd.notna(rec.evi_mean) else np.nan,
            'valid_pixels': int(rec.valid_pixels),
            'total_pixels': int(rec.total_pixels),
            'valid_fraction': float(rec.valid_pixels / rec.total_pixels) if rec.total_pixels else np.nan,
            'is_valid_relaxed': bool(rec.valid_pixels >= MIN_VALID_PIXELS_RELAXED),
            'is_valid_strict': bool(rec.valid_pixels >= MIN_VALID_PIXELS_STRICT),
        })

    partial_obs_df = pd.DataFrame.from_records(records)
    partial_obs_df.to_csv(TABLES_DIR / 'satellite_observations_partial.csv', index=False)
    print(f'  Running total observations: {len(partial_obs_df)}')

obs_df = pd.DataFrame.from_records(records)
obs_df['date'] = pd.to_datetime(obs_df['date'])
obs_df['ndvi_mean'] = pd.to_numeric(obs_df['ndvi_mean'], errors='coerce')
obs_df['evi_mean'] = pd.to_numeric(obs_df['evi_mean'], errors='coerce')
obs_df.loc[~obs_df['is_valid_relaxed'], ['ndvi_mean', 'evi_mean']] = np.nan

label_join_cols = [col for col in ['chain_id', 'is_deciduous', 'deciduous_score', 'leaf_off_start_om', 'full_leaf_off_om', 'leaf_on_return_om'] if col in crowns_sampled.columns]
if label_join_cols:
    obs_df = obs_df.merge(crowns_sampled[label_join_cols].drop_duplicates('chain_id'), on='chain_id', how='left')

obs_df = obs_df.sort_values(['chain_id', 'date']).reset_index(drop=True)
obs_df.to_csv(TABLES_DIR / 'satellite_observations_all.csv', index=False)

coverage_summary = (
    obs_df.groupby('chain_id', as_index=False)
    .agg(
        n_dates=('date', 'nunique'),
        n_valid_relaxed=('is_valid_relaxed', 'sum'),
        n_valid_strict=('is_valid_strict', 'sum'),
        median_valid_fraction=('valid_fraction', 'median'),
    )
    .sort_values('n_valid_strict', ascending=False)
    .reset_index(drop=True)
 )
coverage_summary.to_csv(TABLES_DIR / 'crown_coverage_summary.csv', index=False)

print(f'Wrote {len(obs_df)} sampled crown-date observations.')
print(f'Relaxed-valid observations: {int(obs_df["is_valid_relaxed"].sum())}')
print(f'Strict-valid observations: {int(obs_df["is_valid_strict"].sum())}')
print(f'Saved extraction tables to {TABLES_DIR}')

[001/127] Reading S2B_MSIL2A_20220129T053049_R105_T43RGM_20220213T194758 | date=2022-01-29 | cloud=0.000129


/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_98977/3484009321.py:65: RuntimeWarning: divide by zero encountered in divide
  evi = np.where(invalid, np.nan, 2.5 * (nir_arr - red_arr) / evi_denom)


  Built raster masks for 20 sampled crowns.
  Valid crowns this date: relaxed=11, strict=2
  Running total observations: 20
[002/127] Reading S2B_MSIL2A_20220208T052959_R105_T43RGM_20220220T084206 | date=2022-02-08 | cloud=0.333765


/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_98977/3484009321.py:65: RuntimeWarning: divide by zero encountered in divide
  evi = np.where(invalid, np.nan, 2.5 * (nir_arr - red_arr) / evi_denom)


  Valid crowns this date: relaxed=9, strict=0
  Running total observations: 40
[003/127] Reading S2A_MSIL2A_20220213T052931_R105_T43RGM_20220222T145155 | date=2022-02-13 | cloud=1.3e-05
  Valid crowns this date: relaxed=17, strict=4
  Running total observations: 60
[004/127] Reading S2B_MSIL2A_20220218T052849_R105_T43RGM_20220224T214245 | date=2022-02-18 | cloud=0.000166
  Valid crowns this date: relaxed=11, strict=1
  Running total observations: 80
[005/127] Reading S2B_MSIL2A_20220228T052749_R105_T43RGM_20220303T044843 | date=2022-02-28 | cloud=0.000146
  Valid crowns this date: relaxed=11, strict=1
  Running total observations: 100
[006/127] Reading S2A_MSIL2A_20220305T052721_R105_T43RGM_20220305T193612 | date=2022-03-05 | cloud=6.6e-05
  Valid crowns this date: relaxed=15, strict=2
  Running total observations: 120
[007/127] Reading S2B_MSIL2A_20220310T052649_R105_T43RGM_20220311T030449 | date=2022-03-10 | cloud=28.187707
  Valid crowns this date: relaxed=8, strict=4
  Running tota

In [14]:
strict_df = obs_df[obs_df['is_valid_strict']].copy()
relaxed_df = obs_df[obs_df['is_valid_relaxed']].copy()

join_cols = [col for col in ['chain_id', 'is_deciduous', 'deciduous_score', 'leaf_off_start_om', 'full_leaf_off_om', 'leaf_on_return_om'] if col in crowns_sampled.columns]

weekly_per_crown = (
    strict_df.groupby(['chain_id', 'iso_week'], as_index=False)
    .agg(
        ndvi_weekly_median=('ndvi_mean', 'median'),
        evi_weekly_median=('evi_mean', 'median'),
        n_obs=('ndvi_mean', 'size'),
        n_years=('year', 'nunique'),
    )
    .sort_values(['chain_id', 'iso_week'])
)
if join_cols:
    weekly_per_crown = weekly_per_crown.merge(
        crowns_sampled[join_cols].drop_duplicates('chain_id'),
        on='chain_id',
        how='left',
    )
weekly_per_crown.to_csv(TABLES_DIR / 'weekly_per_crown_strict.csv', index=False)

monthly_per_crown = (
    strict_df.groupby(['chain_id', 'month'], as_index=False)
    .agg(
        ndvi_monthly_median=('ndvi_mean', 'median'),
        evi_monthly_median=('evi_mean', 'median'),
        n_obs=('ndvi_mean', 'size'),
        n_years=('year', 'nunique'),
    )
    .sort_values(['chain_id', 'month'])
)
if join_cols:
    monthly_per_crown = monthly_per_crown.merge(
        crowns_sampled[join_cols].drop_duplicates('chain_id'),
        on='chain_id',
        how='left',
    )
monthly_per_crown.to_csv(TABLES_DIR / 'monthly_per_crown_strict.csv', index=False)

coverage_summary = (
    obs_df.groupby('chain_id', as_index=False)
    .agg(
        n_dates=('date', 'nunique'),
        n_valid_relaxed=('is_valid_relaxed', 'sum'),
        n_valid_strict=('is_valid_strict', 'sum'),
        median_valid_fraction=('valid_fraction', 'median'),
    )
    .sort_values('n_valid_strict', ascending=False)
    .reset_index(drop=True)
)
coverage_summary.to_csv(TABLES_DIR / 'crown_coverage_summary.csv', index=False)

top_crowns = coverage_summary.head(20)['chain_id'].tolist()

fig, ax = plt.subplots(figsize=(8, 4), dpi=180)
ax.hist(coverage_summary['n_valid_strict'], bins=min(20, max(5, coverage_summary['n_valid_strict'].nunique())), color='#457b9d', alpha=0.9)
ax.set_title('Strict valid observation count per crown')
ax.set_xlabel('strict valid dates')
ax.set_ylabel('number of crowns')
ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'coverage_histogram_strict.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(4, 5, figsize=(18, 12), dpi=180, sharex=True, sharey=True)
axes = axes.flatten()
for ax, crown_id in zip(axes, top_crowns):
    crown_obs = relaxed_df[relaxed_df['chain_id'] == crown_id].sort_values('date')
    if crown_obs.empty:
        ax.axis('off')
        continue
    smoothed = crown_obs['ndvi_mean'].rolling(window=3, min_periods=1, center=True).median()
    ax.plot(crown_obs['date'], smoothed, linewidth=1.2, color='#1b7f3b')
    ax.scatter(crown_obs['date'], crown_obs['ndvi_mean'], s=8, alpha=0.45, color='#1b7f3b')
    is_deciduous = crown_obs['is_deciduous'].dropna().iloc[0] if 'is_deciduous' in crown_obs.columns and crown_obs['is_deciduous'].notna().any() else None
    label = 'D' if is_deciduous is True else ('E/O' if is_deciduous is False else '?')
    ax.set_title(f'crown {crown_id} | {label}', fontsize=9)
    ax.grid(True, alpha=0.2)

for ax in axes[len(top_crowns):]:
    ax.axis('off')

fig.suptitle('Per-crown relaxed NDVI trajectories', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(PLOTS_DIR / 'per_crown_relaxed_ndvi_grid.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(4, 5, figsize=(18, 12), dpi=180, sharex=True, sharey=True)
axes = axes.flatten()
for ax, crown_id in zip(axes, top_crowns):
    crown_weekly = weekly_per_crown[weekly_per_crown['chain_id'] == crown_id].sort_values('iso_week')
    if crown_weekly.empty:
        ax.axis('off')
        continue
    ax.plot(crown_weekly['iso_week'], crown_weekly['ndvi_weekly_median'], linewidth=1.2, color='#1b7f3b', label='NDVI')
    ax.plot(crown_weekly['iso_week'], crown_weekly['evi_weekly_median'], linewidth=1.0, color='#d17a22', alpha=0.8, label='EVI')
    is_deciduous = crown_weekly['is_deciduous'].dropna().iloc[0] if 'is_deciduous' in crown_weekly.columns and crown_weekly['is_deciduous'].notna().any() else None
    label = 'D' if is_deciduous is True else ('E/O' if is_deciduous is False else '?')
    ax.set_title(f'crown {crown_id} | {label}', fontsize=9)
    ax.set_xlim(1, 53)
    ax.grid(True, alpha=0.2)

for ax in axes[len(top_crowns):]:
    ax.axis('off')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.suptitle('Per-crown weekly median satellite signals', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOTS_DIR / 'per_crown_weekly_signals_grid.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(4, 5, figsize=(18, 12), dpi=180, sharex=True, sharey=True)
axes = axes.flatten()
for ax, crown_id in zip(axes, top_crowns):
    crown_monthly = monthly_per_crown[monthly_per_crown['chain_id'] == crown_id].sort_values('month')
    if crown_monthly.empty:
        ax.axis('off')
        continue
    ax.plot(crown_monthly['month'], crown_monthly['ndvi_monthly_median'], marker='o', linewidth=1.2, color='#1b7f3b')
    is_deciduous = crown_monthly['is_deciduous'].dropna().iloc[0] if 'is_deciduous' in crown_monthly.columns and crown_monthly['is_deciduous'].notna().any() else None
    label = 'D' if is_deciduous is True else ('E/O' if is_deciduous is False else '?')
    ax.set_title(f'crown {crown_id} | {label}', fontsize=9)
    ax.set_xticks(range(1, 13))
    ax.grid(True, alpha=0.2)

for ax in axes[len(top_crowns):]:
    ax.axis('off')

fig.suptitle('Per-crown monthly NDVI medians', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(PLOTS_DIR / 'per_crown_monthly_ndvi_grid.png', bbox_inches='tight')
plt.close(fig)

print(f'Saved per-crown diagnostics to {PLOTS_DIR}')
print(f'Weekly per-crown rows: {len(weekly_per_crown)} | Monthly per-crown rows: {len(monthly_per_crown)}')
print('Note: these summaries are within-crown medians across repeated observations, not medians across different crowns.')

Saved per-crown diagnostics to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_scl_masked_6May26/plots
Weekly per-crown rows: 266 | Monthly per-crown rows: 101
Note: these summaries are within-crown medians across repeated observations, not medians across different crowns.


In [15]:
feature_source = strict_df.copy()
season_map = {
    'late_winter': [1, 2],
    'pre_monsoon': [3, 4, 5, 6],
    'monsoon': [7, 8, 9],
    'post_monsoon': [10, 11, 12],
}

feature_rows = []
for chain_id, group in feature_source.groupby('chain_id'):
    row = {
        'chain_id': int(chain_id),
        'n_obs_strict': int(len(group)),
        'n_years': int(group['year'].nunique()),
        'ndvi_mean': float(group['ndvi_mean'].mean()),
        'ndvi_std': float(group['ndvi_mean'].std(ddof=0)),
        'ndvi_min': float(group['ndvi_mean'].min()),
        'ndvi_max': float(group['ndvi_mean'].max()),
        'ndvi_amp': float(group['ndvi_mean'].max() - group['ndvi_mean'].min()),
        'evi_mean': float(group['evi_mean'].mean()),
        'evi_std': float(group['evi_mean'].std(ddof=0)),
        'evi_min': float(group['evi_mean'].min()),
        'evi_max': float(group['evi_mean'].max()),
        'evi_amp': float(group['evi_mean'].max() - group['evi_mean'].min()),
        'valid_fraction_median': float(group['valid_fraction'].median()),
    }
    for season_name, months in season_map.items():
        season_group = group[group['month'].isin(months)]
        row[f'ndvi_{season_name}_median'] = float(season_group['ndvi_mean'].median()) if len(season_group) else np.nan
        row[f'evi_{season_name}_median'] = float(season_group['evi_mean'].median()) if len(season_group) else np.nan
    monthly_medians = group.groupby('month')['ndvi_mean'].median()
    for month in range(1, 13):
        row[f'ndvi_month_{month:02d}'] = float(monthly_medians.get(month, np.nan)) if month in monthly_medians.index else np.nan
    feature_rows.append(row)

crown_features = pd.DataFrame(feature_rows)
if 'chain_id' in crowns_filtered.columns:
    join_cols = [col for col in ['chain_id', 'is_deciduous', 'deciduous_score', 'leaf_off_start_om', 'full_leaf_off_om', 'leaf_on_return_om'] if col in crowns_filtered.columns]
    crown_features = crown_features.merge(
        crowns_filtered[join_cols].drop_duplicates('chain_id'),
        on='chain_id',
        how='left',
    )

crown_features = crown_features.sort_values('chain_id').reset_index(drop=True)
crown_features.to_csv(TABLES_DIR / 'crown_level_model_features.csv', index=False)

label_ready = crown_features.dropna(subset=['is_deciduous']) if 'is_deciduous' in crown_features.columns else pd.DataFrame()
summary_stats = {
    'n_crowns_with_features': int(len(crown_features)),
    'n_label_ready_crowns': int(len(label_ready)),
    'strict_valid_observations': int(len(strict_df)),
    'relaxed_valid_observations': int(len(relaxed_df)),
}
(META_DIR / 'feature_summary.json').write_text(json.dumps(summary_stats, indent=2), encoding='utf-8')

print(f'Built crown-level feature table with {len(crown_features)} rows.')
if len(label_ready):
    print('Label-ready crowns by is_deciduous:')
    print(label_ready['is_deciduous'].value_counts(dropna=False).to_dict())
print(f'Saved model features to {TABLES_DIR / "crown_level_model_features.csv"}')

Built crown-level feature table with 12 rows.
Label-ready crowns by is_deciduous:
{False: 6, True: 6}
Saved model features to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_scl_masked_6May26/tables/crown_level_model_features.csv
